# Tass bow softmax workshop With Glove

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import os
import gzip
import shutil
import numpy as np
import pandas as pd
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import ShuffleSplit
from imblearn.over_sampling import RandomOverSampler

DRIVE_PATH = '/content/drive/MyDrive/NLP_TASS'

# 1. Cargar datasets de TASS 2018
data_train = pd.read_csv(os.path.join(DRIVE_PATH, 'tass2018_es_train.csv'))
data_test = pd.read_csv(os.path.join(DRIVE_PATH, 'tass2018_es_test.csv'))

glove_path = os.path.join(DRIVE_PATH, 'glove-sbwc.i25.vec')

gz_path = os.path.join(DRIVE_PATH, 'glove-sbwc.i25.vec.gz')
if not os.path.exists(glove_path) and os.path.exists(gz_path):
    print("Descomprimiendo GloVe desde Drive...")
    with gzip.open(gz_path, 'rb') as f_in:
        with open(glove_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

In [3]:
# 2. Codificar etiquetas (P, N, NEU, NONE -> 0, 1, 2, 3)
le = LabelEncoder()
y_train = le.fit_transform(data_train['sentiment/polarity/value'])
y_test = le.transform(data_test['sentiment/polarity/value'])

In [4]:
# 3. Función para cargar el archivo GloVe en español (.txt)
def load_glove_embeddings(filepath):
    embeddings_dict = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            embeddings_dict[word] = vector
    return embeddings_dict


In [5]:
# Cargar vectores
glove_dict = load_glove_embeddings(glove_path)

In [6]:
# 4. Vectorización ajustada a 300 dimensiones con limpieza de puntuación
def text_to_glove_vector(text, embeddings, embedding_dim=300):
    # Extraer únicamente secuencias alfanuméricas (separa signos de puntuación)
    tokens = re.findall(r'\b\w+\b', str(text).lower())

    # Obtener vectores existentes
    vectors = [embeddings[word] for word in tokens if word in embeddings]

    if len(vectors) > 0:
        return np.mean(vectors, axis=0)

    return np.zeros(embedding_dim)

# Aplicar con 300 dimensiones reales
x_train = np.array([text_to_glove_vector(t, glove_dict, embedding_dim=300) for t in data_train['content']])
x_test = np.array([text_to_glove_vector(t, glove_dict, embedding_dim=300) for t in data_test['content']])

In [7]:
# 5. Sobremuestreo para balancear clases
ros = RandomOverSampler(random_state=1000)
x_train_balanced, y_train_balanced = ros.fit_resample(x_train, y_train)

In [8]:
# 6. Modelo Softmax (Regresión Logística Multinomial)
# Con vectores densos suele funcionar mejor una regularización moderada (C=1.0)
softmax = LogisticRegression(solver='lbfgs', C=1.0, max_iter=1000)

In [11]:
# 7. Validación Cruzada (10-Fold ShuffleSplit)
k_fold = ShuffleSplit(n_splits=10, test_size=0.2, random_state=42)

accuracy_scores = []
recall_scores = []
precision_scores = []
f1_scores = []

for train_idx, val_idx in k_fold.split(x_train_balanced, y_train_balanced):
    x_fold_train, y_fold_train = x_train_balanced[train_idx], y_train_balanced[train_idx]
    x_fold_val, y_fold_val = x_train_balanced[val_idx], y_train_balanced[val_idx]

    softmax.fit(x_fold_train, y_fold_train)
    y_pred_val = softmax.predict(x_fold_val)

    accuracy_scores.append(accuracy_score(y_fold_val, y_pred_val))
    recall_scores.append(recall_score(y_fold_val, y_pred_val, average='macro', zero_division=0))
    precision_scores.append(precision_score(y_fold_val, y_pred_val, average='macro', zero_division=0))
    f1_scores.append(f1_score(y_fold_val, y_pred_val, average='macro', zero_division=0))

In [12]:
# 8. Métricas Promedio de Validación
avg_accuracy = round(np.mean(accuracy_scores) * 100, 2)
avg_recall = round(np.mean(recall_scores) * 100, 2)
avg_precision = round(np.mean(precision_scores) * 100, 2)
avg_f1 = round(np.mean(f1_scores) * 100, 2)

print(f'Averaged over {k_fold.n_splits} folds (macro average):')
print(f'  Accuracy:  {avg_accuracy:.2f}%')
print(f'  Precision: {avg_precision:.2f}%')
print(f'  Recall:    {avg_recall:.2f}%')
print(f'  F1-score:  {avg_f1:.2f}%\n')


Averaged over 10 folds (macro average):
  Accuracy:  63.13%
  Precision: 63.34%
  Recall:    63.21%
  F1-score:  62.97%



In [13]:
# 9. Evaluación Final en el Test Set
softmax.fit(x_train_balanced, y_train_balanced)
y_pred_test = softmax.predict(x_test)

test_f1 = round(f1_score(y_test, y_pred_test, average='macro', zero_division=0) * 100, 2)
test_acc = round(accuracy_score(y_test, y_pred_test) * 100, 2)
test_precision = round(precision_score(y_test, y_pred_test, average='macro', zero_division=0) * 100, 2)
test_recall = round(recall_score(y_test, y_pred_test, average='macro', zero_division=0) * 100, 2)

results = {
    'F1-score': test_f1,
    'Accuracy': test_acc,
    'Precision': test_precision,
    'Recall': test_recall,
}

print('Final Evaluation on Test Set:')
for metric, value in results.items():
    print(f'  {metric}: {value}%')

Final Evaluation on Test Set:
  F1-score: 44.08%
  Accuracy: 49.6%
  Precision: 44.0%
  Recall: 45.34%


**Ventajas de GloVe**

* Captura de relaciones semánticas y sintácticas: Asocia palabras por significado y contexto mediante operaciones vectoriales (por ejemplo, v_rey - v_hombre + v_mujer ≈ v_reina).

* Uso eficiente de estadísticas globales: Combina la factorización de matrices globales con la ventana de contexto local, aprovechando toda la coocurrencia de palabras del corpus.

* Reducción de dimensionalidad: Representa conceptos complejos en vectores densos de baja dimensión (habitualmente 100 o 300 componentes) en lugar de vocabularios dispersos de decenas de miles de columnas.

* Transferencia de conocimiento preentrenado: Permite usar vectores entrenados con miles de millones de palabras, mejorando el rendimiento en tareas con conjuntos de datos pequeños.

* Menor riesgo de sobreajuste (overfitting): Al proyectar las palabras en un espacio vectorial continuo compacto, evita el incremento exponencial de parámetros propio de los modelos basados en la frecuencia directa de términos.

**Desventajas de GloVe**

* Representación estática (indiferente al contexto): Asigna un único vector por palabra, incapaz de diferenciar polisemias u homónimos

* Incapacidad para manejar palabras fuera del vocabulario (Out of Vocabulary - OOV): Si una palabra no estuvo en el corpus de entrenamiento inicial (como errores tipográficos, jergas recientes o nombres propios), no puede generar un vector y la asigna como desconocida o nula.

* Costo computacional y de memoria en el entrenamiento: Requiere construir y almacenar una matriz gigante de coocurrencia global de palabras, lo que consume una cantidad considerable de memoria RAM y tiempo de procesamiento previo.

* Sensible a los sesgos del corpus: Hereda los sesgos culturales, sociales o de género presentes en los textos de origen utilizados durante su entrenamiento.

* Dependencia de la agregación para secuencias: Para representar frases o documentos completos, exige combinar los vectores de las palabras (como promediarlos), lo que destruye el orden sintáctico y la estructura lineal del texto.